# Reading a model's attention

**What you'll learn.** Transformers compute attention on the way to an embedding,
and embpy hands it to you instead of discarding it: which residues the model reads,
how focused each head is, and how that changes with depth.

Attention is `(batch, heads, seq, seq)` — rank 4 — while embpy's output contract
wants `(n_entities, n_dims)`. So the workflow is **extract, then summarise**: the
summaries drop straight into `.obsm` and inherit every exporter.

> **Read these as structural, not causal.** Attention says *what attends to what*.
> Jain & Wallace (NAACL 2019, *Attention is not Explanation*) showed attention can be
> altered substantially without changing predictions, so high attention on a residue
> does not prove it drove the output.

In [ ]:
import numpy as np

from embpy import BioEmbedder, tl

embedder = BioEmbedder(device="auto", organism="human")

# get_model() is the supported way to reach a wrapper for introspection;
# embed() gives you vectors, the wrapper gives you internals.
wrapper = embedder.get_model("esm2_8M")
print(type(wrapper).__name__, "| layers:", wrapper.model.config.num_hidden_layers)

## Extract attention

We use the N-terminus of human HRAS. The **P-loop** (`GAGGVGKS`) binds the
phosphates of GTP — a real, well-localised functional motif to look for.

In [ ]:
seq = "MTEYKLVVVGAGGVGKSALTIQLIQNHFVDEYDPTIEDSYRKQVVIDGETCLLDILDTAGQEEY"
tokens = wrapper.tokenizer(seq, return_tensors="pt")

attn = wrapper.extract_attention(
    tokens["input_ids"],
    attention_mask=tokens["attention_mask"],
    layers=None,          # every layer; pass e.g. [-1] for just the last
)

last = max(attn)
print("layers returned :", sorted(attn))
print("tensor shape    :", tuple(attn[last].shape), "= (batch, heads, seq, seq)")
# tensors come back on the model's device; .cpu() before touching numpy
row_sums = attn[last].sum(-1).cpu().numpy()
print("rows are distributions:", bool(np.allclose(row_sums, 1.0, atol=1e-4)))

Each query row sums to 1 — every residue distributes a fixed budget of attention
over the sequence.

> **Layer indexing.** `extract_attention` returns **one entry per transformer
> block** — no embedding-layer entry, unlike `extract_hidden_states` where index 0
> *is* the embedding layer. Converting between them is what
> `tl.block_to_attention_index` / `tl.block_to_hidden_state_index` are for.

## How focused is each head?

Entropy of a head's attention distribution: low means it concentrates on a few
residues, high means it spreads out. The ceiling is `log(seq_len)`.

In [ ]:
print(f"maximum possible entropy (uniform): {np.log(tokens['input_ids'].shape[1]):.2f}\n")
print(f"{'layer':>5}  {'mean':>6}  {'most focused head':>18}  {'most diffuse head':>18}")
for layer in sorted(attn):
    ent = tl.attention_entropy(attn[layer])     # (batch, n_heads)
    print(f"{layer:>5}  {ent.mean():>6.2f}  {ent.min():>18.2f}  {ent.max():>18.2f}")

A pattern worth noticing: the first block attends broadly across **all** heads,
while later blocks develop a few sharply focused heads alongside diffuse ones —
specialisation emerging with depth.

`head_uniformity` says the same thing on a fixed 0–1 scale, which is easier to
compare across models and sequence lengths:

In [ ]:
for layer in sorted(attn):
    u = tl.head_uniformity(attn[layer])
    print(f"layer {layer}: specialisation {u.mean():.3f}  (0 = uniform, 1 = one-hot)")

## Which residues does the model read?

`received_attention` sums the attention each token *receives*, so it answers
"what is the model looking at?". Let's see the top residues in the final block.

In [ ]:
mass = tl.received_attention(attn[last])[0]      # (seq_len,)

# ESM prepends a <cls> token, so token i+1 corresponds to residue i of `seq`.
top = np.argsort(mass)[::-1][:8]
print(f"{'token':>6}  {'residue':>8}  {'attention':>10}")
for t in top:
    aa = seq[t - 1] if 1 <= t <= len(seq) else "<special>"
    pos = str(t) if aa == "<special>" else f"{aa}{t}"
    print(f"{t:>6}  {pos:>8}  {mass[t]:>10.4f}")

## Attention on a functional motif

`attention_to_gene_set` measures how much of each head's attention lands on a
chosen set of positions. (The name reflects the single-cell case, where tokens are
genes and the set is a pathway; here the set is a structural motif.)

We compare the P-loop against an equally sized stretch of the following helix.

In [ ]:
ploop_start = seq.index("GAGGVGKS")
ploop = [ploop_start + i + 1 for i in range(len("GAGGVGKS"))]   # +1 for <cls>
control = [ploop[-1] + 1 + i for i in range(len(ploop))]        # next 8 residues

print(f"{'layer':>5}  {'P-loop':>8}  {'control':>8}")
for layer in sorted(attn):
    p = tl.attention_to_gene_set(attn[layer], ploop).mean()
    c = tl.attention_to_gene_set(attn[layer], control).mean()
    print(f"{layer:>5}  {p:>8.3f}  {c:>8.3f}")

Both are fractions of total attention mass, so they are directly comparable. Treat
any difference as *structural evidence worth following up*, not proof of
importance — see the caveat at the top.

## Getting attention into an AnnData

The summaries are already `(n_entities, n_dims)`, so they satisfy the output
contract and can be stored and exported like any embedding.

In [ ]:
import anndata as ad
import pandas as pd

adata = ad.AnnData(
    X=np.zeros((1, 1), dtype=np.float32),
    obs=pd.DataFrame({"protein": ["HRAS_fragment"]}, index=["HRAS_fragment"]),
)
adata.obsm["X_attn_entropy"] = tl.attention_entropy(attn[last])
adata.obsm["X_attn_received"] = tl.received_attention(attn[last])

print({k: v.shape for k, v in adata.obsm.items()})

## What you cannot get, and why

Two honest limits:

**Attention-free architectures.** Some models have no attention at all, and embpy
says so rather than inventing something:

In [ ]:
free = embedder.get_model("hyenadna_small_32k", load=False)   # load=False: no download
print("hyenadna has_attention:", free.has_attention, "(implicit long convolution)")

for key in ["morgan_fp", "maccs_fp"]:
    try:
        w = embedder.get_model(key, load=False)
        print(f"{key:20s} has_attention: {w.has_attention}")
    except Exception as exc:
        print(f"{key:20s} unavailable here ({type(exc).__name__})")

**Fused attention kernels.** Models built on `F.scaled_dot_product_attention`,
FlashAttention or Triton never materialise the attention matrix — it exists only
inside the kernel — so no hook can recover it. Extraction would require rewriting
the attention call, which is a model change, not an extraction.

Verified per model in
[the attention-extraction reference](../attention_extraction.md):

| Model | Extractable | Why |
| --- | --- | --- |
| ESM-2, Geneformer | ✅ | HuggingFace `output_attentions` |
| TranscriptFormer | ✅ | explicit `F.softmax` |
| UCE | ✅ | `nn.TransformerEncoderLayer` + `need_weights` pre-hook |
| **scGPT** | ❌ | builds `FlashMHA` unconditionally |
| **STATE** | ❌ | `scaled_dot_product_attention` (fused) |
| HyenaDNA, Caduceus, MiniMol | ❌ | no attention in the architecture |

## Takeaway

- `embedder.get_model(key)` gives you the wrapper; the wrapper gives you internals.
- `extract_attention` returns one entry **per transformer block**.
- `tl.attention_entropy` / `received_attention` / `head_uniformity` /
  `attention_to_gene_set` reduce it to 2-D so it flows through the normal contract.
- Attention is a structural readout. For "what drove this embedding", prefer
  gradient-based attribution.